In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np


In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

print(f"Train samples: {len(train_data)}")
print(f"Test samples:  {len(test_data)}")

100.0%
100.0%
100.0%
100.0%

Train samples: 60000
Test samples:  10000


In [4]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(784, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 784)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = MLP()
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

MLP(
  (fc1): Linear(in_features=784, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=10, bias=True)
  (relu): ReLU()
)

Total parameters: 52,650


In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train(model, loader, optimizer, criterion, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        for images, labels in loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

        acc = 100 * correct / len(loader.dataset)
        print(f"Epoch {epoch+1:2d}/10 | Loss: {total_loss/len(loader):.4f} | Accuracy: {acc:.2f}%")


train(model, train_loader, optimizer, criterion)

# save model weights after training
torch.save(model.state_dict(), 'model.pt')
print("Model saved to weights/model.pt")

Epoch  1/10 | Loss: 0.3261 | Accuracy: 90.48%
Epoch  2/10 | Loss: 0.1465 | Accuracy: 95.69%
Epoch  3/10 | Loss: 0.1086 | Accuracy: 96.71%
Epoch  4/10 | Loss: 0.0855 | Accuracy: 97.39%
Epoch  5/10 | Loss: 0.0723 | Accuracy: 97.70%
Epoch  6/10 | Loss: 0.0614 | Accuracy: 98.04%
Epoch  7/10 | Loss: 0.0532 | Accuracy: 98.27%
Epoch  8/10 | Loss: 0.0460 | Accuracy: 98.52%
Epoch  9/10 | Loss: 0.0406 | Accuracy: 98.66%
Epoch 10/10 | Loss: 0.0357 | Accuracy: 98.76%
Model saved to weights/model.pt


In [6]:
def test(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
    print(f"Test Accuracy: {100 * correct / len(loader.dataset):.2f}%")

In [13]:
def quantize_symmetric(tensor, num_bits=8):
    max_val = tensor.abs().max().item()
    scale   = max_val / (2**(num_bits-1) - 1)
    w_int   = torch.clamp(
                  torch.round(tensor / scale),
                  -(2**(num_bits-1) - 1),
                   (2**(num_bits-1) - 1)
              )
    return w_int.numpy().astype(np.int8), scale

# quantize all layers
w1, s1 = quantize_symmetric(model.fc1.weight.data)
w2, s2 = quantize_symmetric(model.fc2.weight.data)
w3, s3 = quantize_symmetric(model.fc3.weight.data)

In [ ]:
# save weights and scales
import os
os.chdir('/home/shashank/projects/digit-recognition-fpga')
print("Working directory:", os.getcwd())

np.save('weights/w1_int8.npy', w1)
np.save('weights/w2_int8.npy', w2)
np.save('weights/w3_int8.npy', w3)
np.save('weights/scales.npy', np.array([s1, s2, s3]))

# save full model
torch.save(model.state_dict(), 'weights/model.pt')

print(f"W1 shape: {w1.shape} | scale: {s1:.8f}")
print(f"W2 shape: {w2.shape} | scale: {s2:.8f}")
print(f"W3 shape: {w3.shape} | scale: {s3:.8f}")
print("Saved all weights to weights/ folder")


FileNotFoundError: [Errno 2] No such file or directory: 'weights/w1_int8.npy'